<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/Exercises_XP_Day3_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [ ]:
# Optional setup: install dependencies if they are missing in your environment.
# %pip install -q transformers torch


In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Hugging Face transformers make NLP tasks much easier to implement."
print(f"Sentence: {sample_sentence}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Sentence: Hugging Face transformers make NLP tasks much easier to implement.


In [2]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=20,  # Adjusted to accommodate the sentence + special tokens + some padding
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)

index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | hugging      | 17662
    2 | face         |  2227
    3 | transformers | 19081
    4 | make         |  2191
    5 | nl           | 17953
    6 | ##p          |  2361
    7 | tasks        |  8518
    8 | much         |  2172
    9 | easier       |  6082
   10 | to           |  2000
   11 | implement    | 10408
   12 | .            |  1012
   13 | [SEP]        |   102
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (13, '[SEP]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'), (17, '[PAD]'), (18, '[PAD]'), (19, '[PAD]')]


### Exercise 1 reflection
- **Behavior of [CLS] and [SEP]**: The `[CLS]` (Classification) token is added at the beginning of every sequence. In BERT's architecture, the output embedding of this token is used as a summary representation for the entire sequence, which is crucial for tasks like sentiment analysis. The `[SEP]` (Separator) token is used to mark the end of a sequence or to separate two different sentences in tasks like Question Answering.
- **Attention Mask**: The attention mask is a binary tensor (1s for real tokens, 0s for padding). It tells the self-attention mechanism to ignore the `[PAD]` tokens. This ensures that the model doesn't waste computation or let the padding values influence the meaning of the actual words.

## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [3]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "This tutorial is extremely helpful for understanding BERT models!"
prediction = sentiment_pipeline(sentence)
print(f"Sentence: {sentence}")
print(f"Prediction: {prediction}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Sentence: This tutorial is extremely helpful for understanding BERT models!
Prediction: [{'label': 'POSITIVE', 'score': 0.9939714074134827}]


### Exercise 2 reflection
- **Expectation**: The model should ideally return a 'POSITIVE' label given the enthusiastic tone of the sentence.
- **Confidence**: The score (e.g., 0.99) represents the probability the model assigns to the winning label. A higher score indicates the model is very certain about its classification based on the patterns it learned during fine-tuning on the SST-2 dataset.

## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.max_length = max_length
        self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {k: v.to(self.device) for k, v in encoded.items()}

    def predict(self, text: str) -> Dict[str, any]:
        inputs = self.preprocess(text)
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=1)

        score, index = torch.max(probabilities, dim=1)
        label = self.model.config.id2label[index.item()]

        return {"label": label, "probability": score.item()}

In [5]:
analyzer = BERTSentimentAnalyzer()
samples = [
    "I absolutely love how simple it is to use the transformers library!",
    "The documentation was quite confusing and hard to follow initially."
]

for text in samples:
    result = analyzer.predict(text)
    print(f"Text: {text}")
    print(f"Result: {result}\n")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Text: I absolutely love how simple it is to use the transformers library!
Result: {'label': 'POSITIVE', 'probability': 0.9996730089187622}

Text: The documentation was quite confusing and hard to follow initially.
Result: {'label': 'NEGATIVE', 'probability': 0.9996834993362427}



## Exercise 4 - BERT for Named Entity Recognition

### Deliverables reflection
- **Subword handling**: BERT uses WordPiece tokenization. Fragments starting with `##` (like `##p` in `nlp`) are sub-tokens. To handle this, I used the `aggregation_strategy="simple"` in the Hugging Face pipeline, which groups these sub-tokens back into whole words and averages their probability scores to assign a single entity label per word.

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device)
        # Using a pipeline with aggregation_strategy="simple" helps merge subwords automatically
        self.nlp = pipeline("ner", model=self.model, tokenizer=self.tokenizer, device=self.device, aggregation_strategy="simple")

    def recognize(self, text: str):
        return self.nlp(text)

In [7]:
ner = BERTNamedEntityRecognizer()
sample_text = "Google was founded in California by Larry Page and Sergey Brin in 1998."
entities = ner.recognize(sample_text)

print(f"Text: {sample_text}\n")
print("Detected Entities:")
for ent in entities:
    print(f"- {ent['word']}: {ent['entity_group']} (Score: {ent['score']:.4f})")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: Google was founded in California by Larry Page and Sergey Brin in 1998.

Detected Entities:
- Google: ORG (Score: 0.9986)
- California: LOC (Score: 0.9997)
- Larry Page: PER (Score: 0.9997)
- Sergey Brin: PER (Score: 0.9877)


## Exercise 5 - Comparing BERT and GPT

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only (Bidirectional) | Decoder-only (Unidirectional/Autoregressive) |
| Primary purpose | Language Understanding | Language Generation |
| Typical use cases | Sentiment Analysis, NER, QA | Chatbots, Story writing, Code generation |
| Strengths | Considers context from both sides of a word | Creative, fluent, and performs zero-shot tasks |
| Weaknesses | Cannot generate long-form creative text | Can hallucinate and lacks bidirectional context |

## Exercise 6 - BERT inside Retrieval-Augmented Generation

1. **Encoding**: BERT acts as a 'Bi-Encoder'. It processes queries and documents independently to create dense vector representations (embeddings). These vectors capture the semantic meaning rather than just matching keywords.
2. **Vector DB**: These embeddings are stored in a vector database (like Pinecone or FAISS). When a user asks a question, BERT encodes the query, and the system performs a mathematical similarity search (e.g., Cosine Similarity) to find the closest document vectors.
3. **Generative Hand-off**: The most relevant document snippets are retrieved and prepended to the user's prompt. This 'augmented' context is then sent to a generative model like GPT-4 to produce an answer based on specific facts.
4. **Application**: A technical support bot for a company's internal documentation is a perfect use case. BERT finds the right manual page, and GPT explains the solution to the user.